# Day 6: PyTorch Fundamentals

Welcome to Day 6 of the ML-21_DAY_SPRINT! Today we transition from the manual mathematical derivations and raw Python/NumPy implementations of neural networks (Days 2–5) into PyTorch, the industry-standard deep learning framework.

The goal of this notebook is to connect the mathematics of forward passes, backpropagation, and gradient descent to PyTorch's elegant abstractions. We will build our understanding incrementally, ending with a complete end-to-end binary classification project.

## 1. PyTorch Introduction and Imports

We start by importing PyTorch (`torch`) and its essential submodules.

* `torch` is the core library containing tensor operations.
* `torch.nn` contains neural network layers, loss functions, and architectural components.
* `torch.optim` contains optimization algorithms like Stochastic Gradient Descent (SGD) and Adam.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np

print(torch.__version__)

2.11.0+cu130


## 2. Tensor Fundamentals

Tensors are the fundamental data structure in PyTorch. Mathematically, a tensor is a multi-dimensional array. They are analogous to NumPy's ndarrays, but with a critical advantage: they can run on hardware accelerators like GPUs and automatically track operations to calculate derivatives.

We will now explore:
* **Scalar**: 0-dimensional tensor (a single number).
* **Vector**: 1-dimensional tensor.
* **Matrix**: 2-dimensional tensor (representing rows and columns like our feature matrices).
* **Shape and Dimensions**: Using `.ndim` for the number of dimensions and `.shape` (or `.size()`) to inspect size along each dimension.
* **Data Types (dtype)**: Precision of stored numbers, typically 32-bit floating point (`torch.float32`) for neural network weights.
* **Initialization**: Creating tensors filled with zeros, ones, random numbers from a uniform distribution, or standard normal distribution.
* **Indexing and Slicing**: Extracting elements or sub-regions.
* **Reshaping**: Altering the view of a tensor using `.reshape()` or `.view()` without changing its underlying memory.

In [2]:
scalar = torch.tensor(7.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print("Scalar ndim:", scalar.ndim)
print("Scalar shape:", scalar.shape)
print("Scalar dtype:", scalar.dtype)

print("Vector ndim:", vector.ndim)
print("Vector shape:", vector.shape)

print("Matrix ndim:", matrix.ndim)
print("Matrix shape:", matrix.shape)

Scalar ndim: 0
Scalar shape: torch.Size([])
Scalar dtype: torch.float32
Vector ndim: 1
Vector shape: torch.Size([3])
Matrix ndim: 2
Matrix shape: torch.Size([2, 2])


Now we initialize tensors using built-in utility functions.
* `torch.zeros` produces a tensor filled with 0.0, useful for initializing biases.
* `torch.ones` produces a tensor filled with 1.0.
* `torch.rand` draws samples uniformly from the interval [0, 1).
* `torch.randn` draws samples from a standard normal distribution (mean 0, variance 1), which is standard for initializing weights to prevent vanishing/exploding gradients.

In [3]:
zeros_tensor = torch.zeros((2, 3))
ones_tensor = torch.ones((2, 3))
rand_tensor = torch.rand((2, 3))
randn_tensor = torch.randn((2, 3))

print("Zeros:\n", zeros_tensor)
print("Ones:\n", ones_tensor)
print("Uniform Random:\n", rand_tensor)
print("Normal Random:\n", randn_tensor)

Zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
Ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
Uniform Random:
 tensor([[0.6895, 0.4609, 0.8735],
        [0.3552, 0.9643, 0.2546]])
Normal Random:
 tensor([[-0.3120, -1.1437,  0.8599],
        [ 0.4156,  0.2220, -2.9242]])


Next, we practice indexing, slicing, and reshaping. This is essential for selecting batches, extracting specific features, or reshaping image vectors back into multi-channel arrays.

In [4]:
demo_tensor = torch.tensor([[10, 20, 30], [40, 50, 60]])

print("First row:", demo_tensor[0])
print("First element:", demo_tensor[0, 0])
print("Last column:", demo_tensor[:, -1])

reshaped_tensor = demo_tensor.reshape(3, 2)
print("Reshaped to (3, 2):\n", reshaped_tensor)

First row: tensor([10, 20, 30])
First element: tensor(10)
Last column: tensor([30, 60])
Reshaped to (3, 2):
 tensor([[10, 20],
        [30, 40],
        [50, 60]])


## 3. Tensor Mathematical Operations

Neural networks process inputs through continuous linear algebra operations. We must distinguish between element-wise operations and matrix multiplications.

* **Element-wise arithmetic**: Adds, subtracts, or multiplies matching indices. PyTorch overloads `+`, `-`, and `*` for this.
* **Matrix multiplication**: Performs the dot product of rows of the first matrix and columns of the second. In PyTorch, we use `torch.matmul()` or the `@` operator.

*Shape reasoning for Matrix Multiplication*: If matrix $A$ has shape $(M, N)$ and matrix $B$ has shape $(N, P)$, the resulting matrix $C = A \times B$ must have shape $(M, P)$. The inner dimensions must match.

In [5]:
tensor_a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
tensor_b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

elementwise_mult = tensor_a * tensor_b
matrix_mult = tensor_a @ tensor_b

print("Element-wise multiplication:\n", elementwise_mult)
print("Matrix multiplication (@):\n", matrix_mult)

Element-wise multiplication:
 tensor([[ 5., 12.],
        [21., 32.]])
Matrix multiplication (@):
 tensor([[19., 22.],
        [43., 50.]])


## 4. Autograd

Autograd is PyTorch's automatic differentiation engine. It records a graph of all operations executed on tensors that have `requires_grad=True`.

When we compute a scalar output (like a Loss) and call `loss.backward()`, Autograd computes the derivative of the loss with respect to each parameter tensor and stores it in the `.grad` attribute of that tensor. This automates the backpropagation calculus we did by hand on Days 2-5!

Let us evaluate the derivative of a simple function:
$$y = 3x^2 + 5x$$

The analytical derivative with respect to $x$ is:
$$\frac{dy}{dx} = 6x + 5$$

If we evaluate this at $x = 2$:
$$\frac{dy}{dx} = 6(2) + 5 = 17$$

In [6]:
x = torch.tensor(2.0, requires_grad=True)
y = 3 * (x ** 2) + 5 * x

y.backward()

print("Value of x:", x)
print("Value of y:", y)
print("Computed gradient dy/dx at x=2:", x.grad)

Value of x: tensor(2., requires_grad=True)
Value of y: tensor(22., grad_fn=<AddBackward0>)
Computed gradient dy/dx at x=2: tensor(17.)


## 5. Manual Neural Network Calculation using PyTorch Tensors

To bridge the gap between Days 2-5 and PyTorch, let's manually compute a single forward pass, calculate Mean Squared Error (MSE) loss, and compute gradients via Autograd.

Mathematically, the forward pass for a single layer with a sigmoid activation is:
$$\hat{y} = \sigma(X W + b)$$

Where:
* $X$ is our input tensor of shape $(1, 3)$
* $W$ is our weight matrix of shape $(3, 1)$
* $b$ is our bias scalar (shape $(1,)$)
* $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the activation function
* $y$ is the true label (shape $(1, 1)$)
* $Loss = (\hat{y} - y)^2$

In [7]:
X = torch.tensor([[1.5, -0.5, 2.0]])
W = torch.tensor([[0.2], [0.8], [-0.5]], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)
y_true = torch.tensor([[1.0]])

linear_output = X @ W + b
y_pred = torch.sigmoid(linear_output)

loss = (y_pred - y_true) ** 2

loss.backward()

print("Prediction:", y_pred)
print("Loss:", loss)
print("Gradients of Weights (W.grad):\n", W.grad)
print("Gradient of Bias (b.grad):\n", b.grad)

Prediction: tensor([[0.2689]], grad_fn=<SigmoidBackward0>)
Loss: tensor([[0.5344]], grad_fn=<PowBackward0>)
Gradients of Weights (W.grad):
 tensor([[-0.4312],
        [ 0.1437],
        [-0.5749]])
Gradient of Bias (b.grad):
 tensor([-0.2875])


## 6. nn.Module

Instead of managing parameters manually, PyTorch organizes neural networks using `nn.Module`.

* Any custom architecture must inherit from `nn.Module`.
* Inside `__init__()`, we define the layers (such as linear transformations or activations). We must call `super().__init__()` to initialize the base class.
* Inside `forward()`, we define the mathematical sequence of computation. PyTorch calls this method automatically when we pass data to our model instance.

In [8]:
class SimpleNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(3, 1))
        self.bias = nn.Parameter(torch.randn(1))

    def forward(self, x):
        return torch.sigmoid(x @ self.weight + self.bias)

## 7. nn.Linear

Rather than manually defining weight and bias parameters, we use `nn.Linear`. This built-in layer automatically instantiates and manages weights and biases under the hood.

* **Input features**: Number of columns of the input tensor ($X$).
* **Output features**: Number of neurons in this layer.
* **Weight shape**: `(out_features, in_features)` (Note that PyTorch stores weight transposed inside `nn.Linear`).
* **Bias shape**: `(out_features,)`.

Let's construct a linear layer, examine its shapes, and count its parameters.

In [9]:
linear_layer = nn.Linear(in_features=5, out_features=2)

print("Weight shape:", linear_layer.weight.shape)
print("Bias shape:", linear_layer.bias.shape)

total_params = sum(p.numel() for p in linear_layer.parameters())
print("Total parameters:", total_params)

Weight shape: torch.Size([2, 5])
Bias shape: torch.Size([2])
Total parameters: 12


## 8. Activation Functions

Activation functions introduce non-linearity, enabling neural networks to learn complex non-linear patterns. Without activation functions, stacking linear layers would always collapse into a single linear equation ($Y = XW_{eff} + b_{eff}$).

* **ReLU (Rectified Linear Unit)**: $f(x) = \max(0, x)$. It outputs the input if positive, otherwise zero. Helps mitigate vanishing gradient issues.
* **Sigmoid**: $f(x) = \frac{1}{1 + e^{-x}}$. Squashes any input value into the interval $[0, 1]$, making it ideal for predicting probabilities in binary classification.

In [10]:
inputs = torch.tensor([-2.0, -0.5, 0.0, 1.0, 2.0])

relu_activation = nn.ReLU()
sigmoid_activation = nn.Sigmoid()

print("ReLU output:", relu_activation(inputs))
print("Sigmoid output:", sigmoid_activation(inputs))

ReLU output: tensor([0., 0., 0., 1., 2.])
Sigmoid output: tensor([0.1192, 0.3775, 0.5000, 0.7311, 0.8808])


## 9. Loss Functions

A loss function measures how far our predictions are from the ground truth targets.

* **`nn.BCEWithLogitsLoss`**: Used for Binary Classification. It takes raw, un-activated values (logits) and applies a sigmoid internally for numerical stability. Expects targets to be 0 or 1.
* **`nn.MSELoss`**: Used for Regression. Calculates the average squared difference between predictions and targets.
* **`nn.CrossEntropyLoss`**: Used for Multi-class Classification. Takes raw logits of each class and applies a LogSoftmax internally.

In [11]:
logits = torch.tensor([0.5, -1.2, 3.0])
target_binary = torch.tensor([1.0, 0.0, 1.0])

bce_loss_fn = nn.BCEWithLogitsLoss()
loss_val = bce_loss_fn(logits, target_binary)
print("BCEWithLogitsLoss:", loss_val)

BCEWithLogitsLoss: tensor(0.2620)


## 10. Optimizers

Optimizers update model parameters based on the calculated gradients to minimize loss.

* **SGD (Stochastic Gradient Descent)**: Updates parameters in the opposite direction of the gradient scaled by the learning rate $\eta$:
  $$\theta = \theta - \eta \nabla_{\theta} L$$
* **Adam**: Calculates adaptive learning rates for each parameter based on historical first and second moments of the gradients.

Calling `optimizer.step()` performs this mathematical update on all tracked parameters automatically.

In [12]:
dummy_model = nn.Linear(3, 1)
optimizer = optim.SGD(dummy_model.parameters(), lr=0.1)

print("Parameters before step:\n", dummy_model.weight)

loss_value = dummy_model(torch.randn(1, 3)).sum()
loss_value.backward()

optimizer.step()
print("Parameters after step:\n", dummy_model.weight)

Parameters before step:
 Parameter containing:
tensor([[-0.2822, -0.3583,  0.4713]], requires_grad=True)
Parameters after step:
 Parameter containing:
tensor([[-0.0667, -0.3896,  0.3944]], requires_grad=True)


## 11. Complete PyTorch Training Loop Steps

A typical training iteration follows a highly structured, sequential sequence:

1. **Forward Pass**: `outputs = model(inputs)`
2. **Loss Calculation**: `loss = loss_fn(outputs, targets)`
3. **Zero Gradients**: `optimizer.zero_grad()`. This is crucial because PyTorch accumulates gradients on subsequent backward calls by default.
4. **Backward Pass**: `loss.backward()`. Computes gradients using Autograd.
5. **Optimization Step**: `optimizer.step()`. Updates the parameters.

In [13]:
model = nn.Linear(2, 1)
optimizer = optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

inputs = torch.randn(5, 2)
targets = torch.randn(5, 1)

optimizer.zero_grad()
outputs = model(inputs)
loss = loss_fn(outputs, targets)
loss.backward()
optimizer.step()

print("Single step training loss:", loss.item())

Single step training loss: 2.3132426738739014


## 12. Dataset and DataLoader

Instead of feeding the entire dataset to a network at once, we use mini-batches to optimize memory usage and stability.

* **`TensorDataset`**: Wraps raw input and target tensors into a single dataset object.
* **`DataLoader`**: Combines a dataset with a sampler. It automatically handles shuffling (crucial during training to break dataset ordering bias), partitioning into mini-batches (`batch_size`), and multi-process data loading.

In [14]:
X_data = torch.randn(20, 3)
y_data = torch.randn(20, 1)

dataset = TensorDataset(X_data, y_data)
dataloader = DataLoader(dataset, batch_size=5, shuffle=True)

for batch_idx, (batch_X, batch_y) in enumerate(dataloader):
    print(f"Batch {batch_idx} - X shape: {batch_X.shape}, y shape: {batch_y.shape}")

Batch 0 - X shape: torch.Size([5, 3]), y shape: torch.Size([5, 1])
Batch 1 - X shape: torch.Size([5, 3]), y shape: torch.Size([5, 1])
Batch 2 - X shape: torch.Size([5, 3]), y shape: torch.Size([5, 1])
Batch 3 - X shape: torch.Size([5, 3]), y shape: torch.Size([5, 1])


## 13. Train/Validation/Test Split

To ensure our model generalizes, we partition our data into three disjoint subsets:
1. **Train Set**: Used to optimize model parameters.
2. **Validation Set**: Used to tune hyperparameters and monitor overfitting during training.
3. **Test Set**: Used for final evaluation to assess generalization on completely unseen data.

In [15]:
all_features = torch.randn(100, 4)
all_labels = torch.randn(100, 1)

num_train = 70
num_val = 15
num_test = 15

indices = torch.randperm(100)

train_idx = indices[:num_train]
val_idx = indices[num_train:num_train+num_val]
test_idx = indices[num_train+num_val:]

X_train, y_train = all_features[train_idx], all_labels[train_idx]
X_val, y_val = all_features[val_idx], all_labels[val_idx]
X_test, y_test = all_features[test_idx], all_labels[test_idx]

print("Train features shape:", X_train.shape)
print("Validation features shape:", X_val.shape)
print("Test features shape:", X_test.shape)

Train features shape: torch.Size([70, 4])
Validation features shape: torch.Size([15, 4])
Test features shape: torch.Size([15, 4])


## 14. model.train() vs model.eval()

Some PyTorch layers (e.g., Dropout, Batch Normalization) behave differently during training vs evaluation.

* **`model.train()`**: Sets the model in training mode. Enables dropout regularization and uses batch statistics for normalization.
* **`model.eval()`**: Sets the model in evaluation mode. Disables dropout and uses accumulated running statistics for normalization, ensuring deterministic predictions.

In [16]:
test_net = nn.Linear(2, 1)
test_net.train()
print("Is in training mode:", test_net.training)
test_net.eval()
print("Is in training mode:", test_net.training)

Is in training mode: True
Is in training mode: False


## 15. torch.no_grad()

During validation and testing, we only run forward passes to evaluate performance. We do not need to calculate gradients or maintain a computational graph.

Wrapping evaluation code inside `with torch.no_grad():` disables the Autograd engine, which significantly reduces memory consumption and speeds up computation.

In [17]:
eval_x = torch.randn(2, 2)
eval_w = torch.randn(2, 1, requires_grad=True)

with torch.no_grad():
    eval_out = eval_x @ eval_w

print("Output requires grad?", eval_out.requires_grad)

Output requires grad? False


## 16. CPU vs GPU

Deep learning requires massive parallel computation. PyTorch allows us to seamlessly move tensors and models between the CPU and hardware accelerators (GPUs).

We define a generic `device` variable. If a CUDA-enabled GPU is detected, we target `cuda`, otherwise we fall back to `cpu`. We can transfer tensors and models using the `.to(device)` method.

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

sample_tensor = torch.randn(2, 2)
gpu_tensor = sample_tensor.to(device)
print("Tensor device:", gpu_tensor.device)

Using device: cuda
Tensor device: cuda:0


## 17. Evaluation Metrics

Evaluating classification performance requires more than just loss. We evaluate:
* **Accuracy**: Proportion of correct predictions.
* **Precision**: True Positives / (True Positives + False Positives).
* **Recall**: True Positives / (True Positives + False Negatives).
* **F1-score**: Harmonic mean of Precision and Recall.
* **Confusion Matrix**: Structured table showing true vs predicted class associations.

In [19]:
predictions = torch.tensor([1, 0, 1, 1, 0, 1])
targets = torch.tensor([1, 0, 0, 1, 0, 0])

correct_predictions = (predictions == targets).sum().item()
accuracy = correct_predictions / len(targets)

tp = ((predictions == 1) & (targets == 1)).sum().item()
fp = ((predictions == 1) & (targets == 0)).sum().item()
fn = ((predictions == 0) & (targets == 1)).sum().item()
tn = ((predictions == 0) & (targets == 0)).sum().item()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix elements -> TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)

Accuracy: 0.6666666666666666
Precision: 0.5
Recall: 1.0
F1 Score: 0.6666666666666666
Confusion Matrix elements -> TP: 2 FP: 2 FN: 0 TN: 2


## 18. Saving and Loading Models using state_dict

We save trained models using their `state_dict()`. The state dictionary is a standard Python dictionary mapping each layer to its parameter tensors (weights and biases).

* **Saving**: We serialize the `state_dict` to disk using `torch.save()`.
* **Loading**: We instantiate our model class and load the dictionary back into it using `load_state_dict()`.

In [20]:
model_to_save = nn.Linear(3, 1)
torch.save(model_to_save.state_dict(), "model_weights.pth")

new_model = nn.Linear(3, 1)
new_model.load_state_dict(torch.load("model_weights.pth"))
print("Successfully loaded weights into new model!")

Successfully loaded weights into new model!


## 19. Final End-to-End Binary Classification Project

We will now put all these components together to build, train, evaluate, save, and serve a binary classification model.

We will generate synthetic dataset representing a binary classification task with 10 features, split the data, construct custom PyTorch datasets and loaders, build a two-layer neural network with `nn.Linear` and `nn.ReLU`, and optimize it using `nn.BCEWithLogitsLoss` and `optim.Adam`.

### Step 1: Data Preparation and Split
We generate synthetic linear boundaries with noise to create a non-trivial classification dataset.

In [21]:
np.random.seed(42)
X_numpy = np.random.randn(1000, 10).astype(np.float32)
y_numpy = (X_numpy[:, 0] * 1.5 - X_numpy[:, 1] * 2.0 + np.random.randn(1000) * 0.5 > 0).astype(np.float32).reshape(-1, 1)

num_samples = len(X_numpy)
train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)

indices = np.random.permutation(num_samples)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size+val_size]
test_indices = indices[train_size+val_size:]

X_tr, y_tr = torch.tensor(X_numpy[train_indices]), torch.tensor(y_numpy[train_indices])
X_va, y_va = torch.tensor(X_numpy[val_indices]), torch.tensor(y_numpy[val_indices])
X_te, y_te = torch.tensor(X_numpy[test_indices]), torch.tensor(y_numpy[test_indices])

### Step 2: Creating Dataset and DataLoader Instances
We wrap the train, validation, and test sets into `TensorDataset` and `DataLoader` pipelines.

In [22]:
train_ds = TensorDataset(X_tr, y_tr)
val_ds = TensorDataset(X_va, y_va)
test_ds = TensorDataset(X_te, y_te)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

### Step 3: Architecture Definition
We build a custom binary classification model with one hidden layer containing 16 features, passing through a ReLU activation, and terminating in 1 output feature (logits).

In [23]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, 16)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(16, 1)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

### Step 4: Instantiating Model, Loss, Optimizer, and Moving to Device
We instantiate our network and transfer it to our GPU or CPU device, and configure `BCEWithLogitsLoss` and `Adam`.

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BinaryClassifier(input_dim=10).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

### Step 5: Training and Validation Loop
We train for 10 epochs. At the end of each epoch, we track the loss and accuracy of both training and validation sets.

In [30]:
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            loss = loss_fn(logits, batch_y)
            val_loss += loss.item() * batch_X.size(0)

            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == batch_y).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc = correct / len(val_loader.dataset)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 01 | Train Loss: 0.0003 | Val Loss: 1.3557 | Val Acc: 0.9133
Epoch 02 | Train Loss: 0.0003 | Val Loss: 1.3577 | Val Acc: 0.9133
Epoch 03 | Train Loss: 0.0003 | Val Loss: 1.3593 | Val Acc: 0.9133
Epoch 04 | Train Loss: 0.0003 | Val Loss: 1.3598 | Val Acc: 0.9133
Epoch 05 | Train Loss: 0.0003 | Val Loss: 1.3598 | Val Acc: 0.9133
Epoch 06 | Train Loss: 0.0003 | Val Loss: 1.3610 | Val Acc: 0.9133
Epoch 07 | Train Loss: 0.0003 | Val Loss: 1.3630 | Val Acc: 0.9133
Epoch 08 | Train Loss: 0.0003 | Val Loss: 1.3585 | Val Acc: 0.9133
Epoch 09 | Train Loss: 0.0003 | Val Loss: 1.3608 | Val Acc: 0.9133
Epoch 10 | Train Loss: 0.0003 | Val Loss: 1.3626 | Val Acc: 0.9133


### Step 6: Test Set Evaluation
We evaluate the model on the unseen test dataset, measuring predictions, raw probabilities, accuracy, precision, recall, and F1 score.

In [26]:
model.eval()
test_preds = []
test_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        logits = model(batch_X)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(batch_y.cpu().numpy())

test_preds = np.array(test_preds)
test_targets = np.array(test_targets)

correct_count = np.sum(test_preds == test_targets)
test_accuracy = correct_count / len(test_targets)

tp_test = np.sum((test_preds == 1) & (test_targets == 1))
fp_test = np.sum((test_preds == 1) & (test_targets == 0))
fn_test = np.sum((test_preds == 0) & (test_targets == 1))
tn_test = np.sum((test_preds == 0) & (test_targets == 0))

test_precision = tp_test / (tp_test + fp_test) if (tp_test + fp_test) > 0 else 0.0
test_recall = tp_test / (tp_test + fn_test) if (tp_test + fn_test) > 0 else 0.0
test_f1 = 2 * (test_precision * test_recall) / (test_precision + test_recall) if (test_precision + test_recall) > 0 else 0.0

print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1 Score:", test_f1)

Test Accuracy: 0.9133333333333333
Test Precision: 0.8717948717948718
Test Recall: 0.9577464788732394
Test F1 Score: 0.9127516778523489


### Step 7: Saving and Loading the Model
We serialize the final state dict and load it to verify weight consistency.

In [27]:
torch.save(model.state_dict(), "binary_classifier.pth")

eval_model = BinaryClassifier(input_dim=10).to(device)
eval_model.load_state_dict(torch.load("binary_classifier.pth"))
eval_model.eval()
print("Saved and loaded successfully!")

Saved and loaded successfully!


### Step 8: Inference Demonstration
We run inference on a fresh individual sample using our newly loaded model.

In [28]:
fresh_sample = torch.randn(1, 10).to(device)

with torch.no_grad():
    raw_logit = eval_model(fresh_sample)
    probability = torch.sigmoid(raw_logit)
    prediction = (probability >= 0.5).float()

print("Input sample:\n", fresh_sample)
print("Output Logit:", raw_logit.item())
print("Probability:", probability.item())
print("Class Prediction:", prediction.item())

Input sample:
 tensor([[-1.2129,  0.3526,  0.6599, -0.3330, -0.7133,  1.2965,  0.2974,  2.2704,
         -1.9151, -1.2662]], device='cuda:0')
Output Logit: -7.466073989868164
Probability: 0.0005718430620618165
Class Prediction: 0.0


## What I Learned

In this notebook, we transitioned from manually tracking formulas to leveraging PyTorch's automatic execution graph. Here is the fundamental relationship of PyTorch workflows:

`Tensor → Model → Forward Pass → Loss → Autograd → Gradients → Optimizer → Updated Parameters`

### How it connects to Days 2–5 Backpropagation:
* **Days 2–5**: We manually calculated the local derivatives of the activation function and weights using the chain rule, kept track of inputs, and subtracted the gradients multiplied by the learning rate.
* **PyTorch Autograd**: Automates this completely! Whenever we compute calculations on a **Tensor** with `requires_grad=True`, PyTorch records the history as a directed acyclic graph. When we call `Loss.backward()`, it walks backward through the graph automatically computing and caching the exact mathematical chain-rule derivatives in each parameter's `.grad` attribute. Finally, the **Optimizer** updates our model parameters according to our chosen strategy (SGD, Adam, etc.) using those stored gradients.